# Transform TEI <app> elements to CSV values

This notebook is the first step of a process to refactor the DLL's *Library of Digital Latin Texts* encoding guidelines to move away from inline `<app>` elements to standoff markup. There is also a case to be made for producing a more readable format for editors and reviewers to use in the assessment of an edition's apparatus criticus. Also, another DLL-related project involves representing apparatus criticus entries in JSON. Having apparatus criticus data available in a CSV could be helpful for all those uses.

My idea is to use the pilot edition of Calpurnius Siculus' *Bucolica* as a test case. I'll extract the `<app>` entries from it, process them, and return a "long format" CSV with one datapoint per row, according to the following scheme:

```csv
"app_id","id","cts_urn","poem_number","line_number","claim_type","reading_type","reading","supporting_evidence","note","reporting_rank"
"33bb00fc-9567-4759-9963-93bb5c883f86","b2e9bb50-da27-411b-a9d2-10741f35f52f","urn:cts:latinLit:phi0830.phi001.dll:1.1@C.","1","1","reading","lemma","C.","G P A φ Ulit. Wernsd.","Glaeser sqq.",""
"33bb00fc-9567-4759-9963-93bb5c883f86","575478bc-9438-4292-87e9-8a660090bea6","urn:cts:latinLit:phi0830.phi001.dll:1.1@C.","1","1","reading","variant","","N π χ p","","1"
"33bb00fc-9567-4759-9963-93bb5c883f86","f3029bb5-ce04-4452-99b6-29906840b5cc","urn:cts:latinLit:phi0830.phi001.dll:1.1@C.","1","1","reading","variant","O.","ε β γ μ ρ","edd. ante Glaeser","2"
"79fcedbd-0311-494c-a783-86d0bd2656a9","f1d19109-b50d-427a-9817-135014ce6c49","urn:cts:latinLit:phi0830.phi001.dll:1.1@Nondum","1","1","reading","lemma","Nondum","","",""
"79fcedbd-0311-494c-a783-86d0bd2656a9","ac4f112e-f62f-4f98-a774-d0fcecd268a7","urn:cts:latinLit:phi0830.phi001.dll:1.1@Nondum","1","1","reading","variant","nundum","G P","","1"
```

Here's a rundown of the columns:

- `app_id`: a UUID4 ID for one `<app>` element. This gives me a way of relating all the rows for an `<app>` to each other.
- `id`: a UUID4 ID for a `<lem>`, `<rdg>`, or `<note>` element inside the `<app>`.
- `cts_urn`: The CTS URN of the author, work, and lemma in an `<app>`.
- `poem_number`: The number of the poem, extracted from the `@n` value of `<div>` with `@subtype="poem"`. (This should probably be "section_number", come to think of it.)
- `line_number`: The line number of the poem, extracted from the `@n` value of `<l>`.
- `claim_type`: This will almost always be `apparatus_entry`, but it could be extended to other terms.
- `reading_type`: Either `reading` or `variant`.
- `reading`: The text of a `<lem>` or `<rdg>`.
- `supporting_evidence`: The values of `@wit` and `@source` on `<lem>` or `<rdg>`.
- `note`: The value of any `<note>` associated with a `<lem>` or `<rdg>`.
- `reporting_rank`: The order in which the editor reported a `<rdg>` (`<lem>` gets "").

The goal is to read in an existing TEI XML file, extract all the relevant elements, and use the values to populate a CSV file.

In [1]:
# Import the necessary libraries
import lxml.etree as ET
import pandas as pd
import csv
from uuid import uuid4

In [4]:
# Parse the XML file and create an ElementTree object
tree = ET.parse("../xml/editio.xml")

# Register a namespace for the TEI elements
ns = {"tei": "http://www.tei-c.org/ns/1.0"}

In [5]:
# Functions
def extract_reading_text(element):
    """Extract text, following lemmas in nested apparatus entries."""
    if element is None:
        return ""

    parts = [element.text or ""]

    for child in element:
        # Skip comments and processing instructions
        if not isinstance(child.tag, str):
            parts.append(child.tail or "")
            continue

        if ET.QName(child).localname == "app":
            nested_lemma = child.find("tei:lem", namespaces=ns)
            parts.append(extract_reading_text(nested_lemma))
        else:
            parts.append(extract_reading_text(child))

        parts.append(child.tail or "")

    return normalize_space("".join(parts))


def extract_lemma(app):
    """Extract the lemma from a <lem> element."""
    lemma = app.find("tei:lem", namespaces=ns)
    return extract_reading_text(lemma)


def extract_lem_rdg_references(
    element: ET._Element | None,
) -> str | list[str]:
    """
    Extract normalized `wit` and `source` references from a <lem> or <rdg>.

    Leading '#' characters are removed, and whitespace-separated references
    are returned as lists. Missing attributes produce empty lists.
    """
    if element is None:
        return []

    element_name = ET.QName(element).localname
    if element_name not in {"lem", "rdg"}:
        raise ValueError(
            f"Expected a <lem> or <rdg> element, received <{element_name}>"
        )

    def normalize(attribute: str) -> list[str]:
        value = element.get(attribute, "")
        return [token.removeprefix("#") for token in value.split()]

    wits = normalize("wit")
    sources = normalize("source")
    wits_and_sources = wits + sources
    supporting_evidence = " ".join(wits_and_sources) if wits_and_sources else ""
    return supporting_evidence


def normalize_space(value: str | None) -> str:
      """Equivalent to XPath normalize-space()."""
      return " ".join((value or "").split())


def extract_note(note: ET._Element | None) -> str | None:
    if note is None:
        return None

    return normalize_space("".join(note.itertext()))


def make_cts_urn(cts_base, poem_number: str, line_number: str, lemma) -> str:
    """Return a CTS URN for a given poem and line number."""
    return f"{cts_base}:{poem_number}.{line_number}@{lemma}"


def extract_immediately_following_note(
    element: ET._Element | None,
    namespaces: dict[str, str],
) -> ET._Element | None:
    """Return a <note> only when it is the next element sibling."""
    if element is None:
        return None

    notes = element.xpath(
        "following-sibling::*[1][self::tei:note]",
        namespaces=namespaces,
    )
    return notes[0] if notes else None

In the next cell, I open a blank CSV file, loop through XML, and write the relevant information to rows.

In [6]:
with open("CSV/calpsicapp.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, delimiter=",", quotechar='"', quoting=csv.QUOTE_ALL)
    print("Processing apps...")

    writer.writerow(
        ["app_id", "id", "cts_urn", "poem_number", "line_number", "claim_type", "reading_type", "reading", "supporting_evidence", "note", "reporting_rank"]
    )

    # Iterate over each poem in the XML tree
    for poem in tree.xpath("//tei:div[@subtype='poem']", namespaces=ns):
        # Get the poem number and iterate over each line in the poem
        poem_number = poem.get("n")
        for line in poem.findall("tei:lg/tei:l", namespaces=ns):
            line_number = line.get("n")
            print(f"Processing line {line_number} of poem {poem_number}...")
            # Iterate over each <app> element in the line
            for app in line.xpath(".//tei:app", namespaces=ns):
                app_id = str(uuid4())
                if app.find("tei:lem", namespaces=ns) is None:
                    continue
                else:
                    print(f"Processing a lemma …")
                    id = str(uuid4())
                    lemma_element = app.find("tei:lem", namespaces=ns)
                    lemma_note_element = extract_immediately_following_note(lemma_element, namespaces=ns)
                    claim_type = "reading"
                    reading_type = "lemma"
                    lemma = extract_lemma(app)
                    cts_urn = make_cts_urn(
                        "urn:cts:latinLit:phi0830.phi001.dll",
                        poem_number,
                        line_number,
                        lemma,
                    )
                    supporting_evidence = extract_lem_rdg_references(lemma_element)
                    note = extract_note(lemma_note_element)
                    # Write the lemma and its supporting evidence to the CSV file
                    print(f"Writing lemma: {lemma} with supporting evidence: {supporting_evidence} and note: {note}")
                    writer.writerow(
                        [
                            app_id,
                            id,
                            cts_urn,
                            poem_number,
                            line_number,
                            claim_type,
                            reading_type,
                            lemma,
                            supporting_evidence,
                            note,
                            "",]
                    )
                
                # Iterate over each <rdg> element in the app
                readings = app.findall("tei:rdg", namespaces=ns)
                print(f"Processing {len(readings)} readings...")
                for i, rdg in enumerate(readings, start=1):
                    print(f"Processing reading {i}...")
                    id = str(uuid4())
                    claim_type = "reading"
                    reading_type = "variant"
                    reading = extract_reading_text(rdg)
                    cts_urn = make_cts_urn(
                        "urn:cts:latinLit:phi0830.phi001.dll",
                        poem_number,
                        line_number,
                        lemma,
                    )
                    supporting_evidence = extract_lem_rdg_references(rdg)
                    rdg_note_element = extract_immediately_following_note(rdg, namespaces=ns)
                    note = extract_note(rdg_note_element)
                    # Write the reading and its supporting evidence to the CSV file
                    print(f"Writing reading: {reading} with supporting evidence: {supporting_evidence} and note: {note}")
                    writer.writerow(
                        [
                            app_id,
                            id,
                            cts_urn,
                            poem_number,
                            line_number,
                            claim_type,
                            reading_type,
                            reading,
                            supporting_evidence,
                            note,
                            i]
                    )

Processing apps...
Processing line 1 of poem 1...
Processing a lemma …
Writing lemma: C. with supporting evidence: G P A φ Ulit. Wernsd. and note: Glaeser sqq.
Processing 2 readings...
Processing reading 1...
Writing reading:  with supporting evidence: N π χ p and note: None
Processing reading 2...
Writing reading: O. with supporting evidence: ε β γ μ ρ and note: edd. ante Glaeser
Processing a lemma …
Writing lemma: Nondum with supporting evidence:  and note: None
Processing 1 readings...
Processing reading 1...
Writing reading: nundum with supporting evidence: G P and note: None
Processing a lemma …
Writing lemma: declinis with supporting evidence: N Heins. Schenkl and note: None
Processing 3 readings...
Processing reading 1...
Writing reading: declivis with supporting evidence: G V edd. and note: None
Processing reading 2...
Writing reading: declivus with supporting evidence: P p and note: None
Processing reading 3...
Writing reading: declives with supporting evidence: μ and note: No